In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/KB_FinalProject_Team6/완전쓰는거_ver2/인허가_master.csv")

In [ ]:
df.head()

In [ ]:
import pandas as pd

industry_cols = [
    col for col in df.columns
    if col.startswith("업종__")
]

base_cols = [
    col for col in df.columns
    if col not in industry_cols
]

print(f"전체 행 수: {len(df):,}")
print(f"전체 컬럼 수: {df.shape[1]}")
print(f"일반 컬럼 수: {len(base_cols)}")
print(f"업종별 0/1 컬럼 수: {len(industry_cols)}")

structure = pd.DataFrame({
    "자료형": df.dtypes.astype(str),
    "고유값수": df.nunique(dropna=True)
}).rename_axis("컬럼명")

# 일반 컬럼의 구조 확인
with pd.option_context("display.max_rows", None):
    display(structure.loc[base_cols])

# 업종별 컬럼의 자료형 확인
print("업종별 컬럼 자료형:")
display(df[industry_cols].dtypes.astype(str).value_counts())

In [ ]:
missing_counts = df.isna().sum()

text_cols = df.select_dtypes(
    include=["object", "string", "category"]
).columns

for col in text_cols:
    blank_mask = (
        df[col].astype("string")
        .str.strip()
        .eq("")
        .fillna(False)
    )
    missing_counts[col] += int(blank_mask.sum())

missing_report = pd.DataFrame({
    "결측수": missing_counts,
    "결측률(%)": (missing_counts / len(df) * 100).round(2)
}).rename_axis("컬럼명")

missing_report = missing_report.sort_values(
    "결측수",
    ascending=False
)

print(f"결측이 있는 컬럼 수: {(missing_counts > 0).sum()}개")

with pd.option_context("display.max_rows", None):
    display(missing_report[missing_report["결측수"] > 0])

In [ ]:
# 모든 컬럼의 값이 동일한 중복 행
duplicate_rows = df.duplicated(keep="first")

# 점포 식별자 확인
ids = df["통합점포ID"].astype("string").str.strip()
valid_ids = ids.notna() & ids.ne("")

# 결측·공란은 제외하고 중복 검사
duplicate_ids = valid_ids & ids.duplicated(keep=False)

duplicate_report = pd.DataFrame({
    "검사 항목": [
        "전체 값이 동일한 중복 초과 행 수",
        "통합점포ID 결측·공란 행 수",
        "통합점포ID 고유값 수",
        "중복된 통합점포ID 종류 수",
        "통합점포ID 중복에 해당하는 행 수"
    ],
    "결과": [
        int(duplicate_rows.sum()),
        int((~valid_ids).sum()),
        ids[valid_ids].nunique(),
        ids[duplicate_ids].nunique(),
        int(duplicate_ids.sum())
    ]
})

display(duplicate_report)

# 중복된 점포가 있으면 확인
if duplicate_ids.any():
    preview_cols = [
        col for col in ["통합점포ID", "점포이력ID", "사업장명"]
        if col in df.columns
    ]
    display(df.loc[duplicate_ids, preview_cols].head(20))

In [ ]:
# 행정동코드는 범주를 나타내는 코드
df["행정동코드"] = df["행정동코드"].astype("string")

# 업종 컬럼이 모두 0/1인지 확인한 뒤 작은 정수형으로 변경
if not df[industry_cols].isin([0, 1]).all().all():
    raise ValueError("업종 컬럼에 결측 또는 0/1 이외의 값이 있습니다.")

df[industry_cols] = df[industry_cols].astype("uint8")

In [ ]:
import pandas as pd
import plotly.express as px

industry_cols = [
    col for col in df.columns
    if col.startswith("업종__")
]

# EDA 1단계에서 한 행이 한 점포임을 확인
total_stores = len(df)

def plot_store_counts(table, category, title, height=600):
    fig = px.bar(
        table,
        x="점포수",
        y=category,
        orientation="h",
        text="점포수",
        hover_data={"전체점포대비비중(%)": ":.2f"},
        title=title,
        template="plotly_white",
        color_discrete_sequence=["#4472C4"]
    )

    fig.update_traces(
        texttemplate="%{x:,.0f}",
        textposition="outside",
        cliponaxis=False
    )

    fig.update_layout(
        height=height,
        font=dict(
            family="Malgun Gothic, Apple SD Gothic Neo, sans-serif",
            size=12
        ),
        yaxis=dict(
            categoryorder="total ascending",
            automargin=True
        ),
        xaxis=dict(tickformat=",d"),
        margin=dict(r=100)
    )

    fig.show(renderer="colab")

In [ ]:
# 합산 전에 업종 컬럼 값 확인
if not industry_cols:
    raise ValueError("업종별 0/1 컬럼이 없습니다.")

if not df[industry_cols].isin([0, 1]).all().all():
    raise ValueError("업종 컬럼에 결측 또는 0/1 이외 값이 있습니다.")

# 업종별 점포 수
industry_dist = (
    df[industry_cols]
    .sum()
    .rename_axis("업종")
    .reset_index(name="점포수")
)

industry_dist["업종"] = (
    industry_dist["업종"].str.removeprefix("업종__")
)

industry_dist["점포수"] = industry_dist["점포수"].astype("int64")

industry_dist["전체점포대비비중(%)"] = (
    industry_dist["점포수"] / total_stores * 100
).round(2)

industry_dist = industry_dist.sort_values(
    "점포수", ascending=False
).reset_index(drop=True)

# 복합업종 점포 수
industry_count_per_store = df[industry_cols].sum(axis=1)

print(f"전체 점포 수: {total_stores:,}개")
print(f"업종 종류 수: {len(industry_cols)}개")
print(
    f"2개 이상 업종을 가진 점포: "
    f"{industry_count_per_store.gt(1).sum():,}개"
)

# 상위 20개 확인
display(industry_dist.head(20))

plot_store_counts(
    industry_dist.head(20),
    category="업종",
    title="업종별 점포 수 상위 20개",
    height=750
)

In [ ]:
# 분석용 지역 값 정리: 기존 df는 변경하지 않음
district = (
    df["구군"]
    .astype("string")
    .str.strip()
    .fillna("미상")
    .replace("", "미상")
)

district_dist = (
    district.value_counts()
    .rename_axis("구군")
    .reset_index(name="점포수")
)

district_dist["전체점포대비비중(%)"] = (
    district_dist["점포수"] / total_stores * 100
).round(2)

display(district_dist)

print(f"구군별 점포 수 합계: {district_dist['점포수'].sum():,}개")

plot_store_counts(
    district_dist,
    category="구군",
    title="구군별 점포 수",
    height=600
)

In [ ]:
import pandas as pd
import plotly.express as px

date_cols = [
    "최초인허가일",
    "최근인허가일",
    "폐업일자_dt"
]

df_dates = df[
    ["통합점포ID", "영업여부"] + date_cols
].copy()

date_summary = []

for col in date_cols:
    raw = (
        df[col].astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    parsed = pd.to_datetime(
        raw,
        format="%Y-%m-%d",
        errors="coerce"
    )

    failed = raw.notna() & parsed.isna()
    df_dates[col] = parsed

    date_summary.append({
        "컬럼": col,
        "유효 날짜 수": int(parsed.notna().sum()),
        "결측수": int(raw.isna().sum()),
        "변환 실패 수": int(failed.sum()),
        "최소일": parsed.min(),
        "최대일": parsed.max()
    })

    if failed.any():
        print(f"{col} 변환 실패 예시:")
        display(df.loc[failed, ["통합점포ID", col]].head(10))

date_summary = pd.DataFrame(date_summary)
display(date_summary)

if date_summary["변환 실패 수"].sum() > 0:
    raise ValueError("날짜 변환 실패 값을 먼저 확인해주세요.")

# 날짜 순서와 영업 상태의 일관성 확인
date_checks = pd.DataFrame({
    "검사 항목": [
        "최근인허가일이 최초인허가일보다 빠름",
        "폐업일이 최근인허가일보다 빠름",
        "폐업 상태(F)인데 폐업일이 없음",
        "영업 상태(T)인데 폐업일이 있음"
    ],
    "행 수": [
        int((df_dates["최근인허가일"] < df_dates["최초인허가일"]).sum()),
        int((df_dates["폐업일자_dt"] < df_dates["최근인허가일"]).sum()),
        int((
            df_dates["영업여부"].eq("F")
            & df_dates["폐업일자_dt"].isna()
        ).sum()),
        int((
            df_dates["영업여부"].eq("T")
            & df_dates["폐업일자_dt"].notna()
        ).sum())
    ]
})

display(date_checks)

In [ ]:
chart_font = dict(
    family="Malgun Gothic, Apple SD Gothic Neo, sans-serif",
    size=12
)

yearly_tables = {}

targets = {
    "최초인허가일": "최초인허가 연도별 점포 수",
    "폐업일자_dt": "폐업기록 연도별 점포 수"
}

for col, title in targets.items():
    yearly = (
        df_dates[col].dropna()
        .dt.year
        .value_counts()
        .sort_index()
        .rename_axis("연도")
        .reset_index(name="점포수")
    )

    yearly_tables[col] = yearly

    print(title)
    display(yearly.tail(10))

    fig = px.bar(
        yearly,
        x="연도",
        y="점포수",
        title=title,
        template="plotly_white",
        color_discrete_sequence=["#4472C4"]
    )

    fig.update_layout(
        font=chart_font,
        height=400,
        yaxis=dict(tickformat=",d")
    )

    fig.update_xaxes(tickformat="d")
    fig.show(renderer="colab")

In [ ]:
month_index = pd.period_range(
    start="2023-01",
    end="2026-10",
    freq="M"
)

monthly_dist = pd.DataFrame(index=month_index)

monthly_targets = {
    "최초인허가일": "최초인허가점포수",
    "폐업일자_dt": "폐업기록점포수"
}

for col, output_name in monthly_targets.items():
    counts = (
        df_dates[col].dropna()
        .dt.to_period("M")
        .value_counts()
    )

    # 자료에 해당 월의 기록이 없으면 0으로 표시
    monthly_dist[output_name] = (
        counts.reindex(month_index, fill_value=0)
        .astype("int64")
    )

monthly_dist.index = monthly_dist.index.to_timestamp()
monthly_dist = monthly_dist.rename_axis("월").reset_index()

# 최근 12개월 표
display(monthly_dist.tail(12))

fig = px.line(
    monthly_dist,
    x="월",
    y=["최초인허가점포수", "폐업기록점포수"],
    markers=True,
    title="월별 최초인허가·폐업 기록 분포",
    labels={
        "value": "점포수",
        "variable": "기록 종류"
    },
    template="plotly_white"
)

fig.update_layout(
    font=chart_font,
    height=500,
    yaxis=dict(tickformat=",d"),
    legend_title_text=""
)

fig.update_xaxes(
    dtick="M3",
    tickformat="%Y-%m"
)

fig.show(renderer="colab")

In [ ]:
import pandas as pd
import plotly.express as px

industry_cols = [
    col for col in df.columns
    if col.startswith("업종__")
]

status_order = ["영업", "폐업기록", "상태미확인"]

status_code = (
    df["영업여부"]
    .astype("string")
    .str.strip()
    .str.upper()
)

status = status_code.map({
    "T": "영업",
    "F": "폐업기록"
}).fillna("상태미확인")

district = (
    df["구군"]
    .astype("string")
    .str.strip()
    .fillna("미상")
    .replace("", "미상")
)

# 상태별 집계표에 점포 수와 폐업 비중 추가
def summarize_status(counts):
    counts = counts.reindex(
        columns=status_order,
        fill_value=0
    ).astype("int64")

    known = counts["영업"] + counts["폐업기록"]

    return pd.DataFrame({
        "전체점포수": counts.sum(axis=1),
        "영업점포수": counts["영업"],
        "폐업기록점포수": counts["폐업기록"],
        "상태미확인수": counts["상태미확인"],
        "상태확인점포수": known,
        "기록상폐업비중(%)": (
            counts["폐업기록"]
            / known.where(known > 0)
            * 100
        ).round(2)
    })

overall_summary = summarize_status(
    status.value_counts().to_frame().T
)
overall_summary.index = ["전체"]

overall_share = overall_summary["기록상폐업비중(%)"].iloc[0]

display(overall_summary)

In [ ]:
# 이후 업종별 분석에서도 사용할 그래프 함수
def plot_closure_share(table, category, title, height=600):
    fig = px.bar(
        table,
        x="기록상폐업비중(%)",
        y=category,
        orientation="h",
        text="기록상폐업비중(%)",
        hover_data=[
            "전체점포수",
            "영업점포수",
            "폐업기록점포수",
            "상태미확인수"
        ],
        title=title,
        template="plotly_white",
        color_discrete_sequence=["#4472C4"]
    )

    fig.update_traces(
        texttemplate="%{x:.2f}%",
        textposition="outside",
        cliponaxis=False
    )

    if pd.notna(overall_share):
        fig.add_vline(
            x=overall_share,
            line_dash="dash",
            line_color="#B54747",
            annotation_text=f"전체 {overall_share:.2f}%"
        )

    fig.update_layout(
        height=height,
        font=dict(
            family="Malgun Gothic, Apple SD Gothic Neo, sans-serif",
            size=12
        ),
        yaxis=dict(
            categoryorder="total ascending",
            automargin=True
        ),
        margin=dict(r=100)
    )

    fig.update_xaxes(ticksuffix="%", rangemode="tozero")
    fig.show(renderer="colab")


district_counts = pd.crosstab(district, status)

district_status = (
    summarize_status(district_counts)
    .rename_axis("구군")
    .reset_index()
    .sort_values("기록상폐업비중(%)", ascending=False)
    .reset_index(drop=True)
)

display(district_status)

plot_closure_share(
    district_status,
    category="구군",
    title="구군별 기록상 폐업 비중"
)

In [ ]:
# 업종별로 각 상태에 해당하는 점포 수 합산
industry_counts = pd.DataFrame({
    state: df.loc[
        status.eq(state), industry_cols
    ].sum()
    for state in status_order
})

industry_counts.index = (
    industry_counts.index.str.removeprefix("업종__")
)

industry_status = (
    summarize_status(industry_counts)
    .rename_axis("업종")
    .reset_index()
    .sort_values("전체점포수", ascending=False)
    .reset_index(drop=True)
)

# 표본이 많은 업종부터 비교
industry_top20 = industry_status.head(20).copy()

display(industry_top20)

plot_closure_share(
    industry_top20,
    category="업종",
    title="점포 수 상위 20개 업종의 기록상 폐업 비중",
    height=800
)

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

industry_cols = [
    col for col in df.columns
    if col.startswith("업종__")
]

district = (
    df["구군"].astype("string")
    .str.strip()
    .fillna("미상")
    .replace("", "미상")
)

status_code = (
    df["영업여부"].astype("string")
    .str.strip()
    .str.upper()
)

# 구군은 전체 점포 수가 많은 순서로 배치
regions = district.value_counts().index.tolist()

# 전체·영업·폐업 기록을 각각 집계
matrices = {}

for name, code in [
    ("전체점포수", None),
    ("영업점포수", "T"),
    ("폐업기록점포수", "F")
]:
    selected = (
        pd.Series(True, index=df.index)
        if code is None
        else status_code.eq(code).fillna(False)
    )

    matrix = (
        df.loc[selected, industry_cols]
        .groupby(district[selected])
        .sum()
        .reindex(
            index=regions,
            columns=industry_cols,
            fill_value=0
        )
        .astype("int64")
    )

    matrix.columns = matrix.columns.str.removeprefix("업종__")
    matrices[name] = matrix

known = matrices["영업점포수"] + matrices["폐업기록점포수"]

matrices["상태확인점포수"] = known
matrices["상태미확인수"] = matrices["전체점포수"] - known

rate = (
    matrices["폐업기록점포수"]
    / known.where(known > 0)
    * 100
)

matrices["기록상폐업비중(%)"] = rate

# 구군×업종을 한 행씩 나타내는 표
cross_status = pd.concat({
    name: matrix.rename_axis(
        index="구군", columns="업종"
    ).stack()
    for name, matrix in matrices.items()
}, axis=1).reset_index()

# 같은 업종의 전체 비중을 비교 기준으로 추가
industry_overall = (
    matrices["폐업기록점포수"].sum()
    / known.sum().where(known.sum() > 0)
    * 100
)

cross_status["업종전체폐업비중(%)"] = (
    cross_status["업종"].map(industry_overall)
)

cross_status["업종전체대비차이(%p)"] = (
    cross_status["기록상폐업비중(%)"]
    - cross_status["업종전체폐업비중(%)"]
)

MIN_N = 100  # 비교용 표본 수 기준: 필요하면 변경
cross_status["비교대상"] = (
    cross_status["상태확인점포수"] >= MIN_N
)

print(f"전체 구군×업종 조합: {len(cross_status):,}개")
print(f"확인 점포 {MIN_N}개 이상 조합: {cross_status['비교대상'].sum():,}개")

display(cross_status.head(10).round(2))

In [ ]:
TOP_N = 10

# 전체 점포 수가 많은 업종 선택
top_industries = (
    matrices["전체점포수"]
    .sum()
    .nlargest(TOP_N)
    .index.tolist()
)

# 행: 업종 / 열: 구군
plot_rate = rate.loc[regions, top_industries].T
plot_known = known.loc[regions, top_industries].T
plot_closed = (
    matrices["폐업기록점포수"]
    .loc[regions, top_industries].T
)
plot_total = (
    matrices["전체점포수"]
    .loc[regions, top_industries].T
)

# 표본 수가 적은 셀은 그림에서만 숨김
plot_rate = plot_rate.where(plot_known >= MIN_N)
z = plot_rate.to_numpy(dtype=float)

text = [
    [f"{value:.1f}%" if pd.notna(value) else "" for value in row]
    for row in z
]

customdata = np.stack([
    plot_total.to_numpy(),
    plot_closed.to_numpy(),
    plot_known.to_numpy()
], axis=-1)

fig = go.Figure(go.Heatmap(
    z=z,
    x=plot_rate.columns.tolist(),
    y=plot_rate.index.tolist(),
    zmin=0,
    zmax=100,
    colorscale=[
        [0, "#F0F6FC"],
        [1, "#4472C4"]
    ],
    text=text,
    texttemplate="%{text}",
    textfont=dict(size=10, color="black"),
    customdata=customdata,
    hoverongaps=False,
    hovertemplate=(
        "구군: %{x}<br>"
        "업종: %{y}<br>"
        "전체 점포: %{customdata[0]:,d}개<br>"
        "폐업 기록: %{customdata[1]:,d}개<br>"
        "상태 확인 점포: %{customdata[2]:,d}개<br>"
        "기록상 폐업 비중: %{z:.2f}%"
        "<extra></extra>"
    ),
    colorbar=dict(title="폐업 비중(%)")
))

fig.update_layout(
    title=f"구군×업종별 기록상 폐업 비중 — 확인 점포 {MIN_N}개 이상",
    height=650,
    template="plotly_white",
    plot_bgcolor="#E5E7EB",
    font=dict(
        family="Malgun Gothic, Apple SD Gothic Neo, sans-serif",
        size=12
    ),
    xaxis=dict(title="구군", tickangle=-45),
    yaxis=dict(
        title="업종",
        autorange="reversed",
        automargin=True
    )
)

fig.show(renderer="colab")

In [ ]:
target_industry = "일반음식점"
# 예: "건강기능식품일반판매업", "휴게음식점"

industry_detail = (
    cross_status.loc[
        cross_status["업종"].eq(target_industry)
    ]
    .sort_values("기록상폐업비중(%)", ascending=False)
    .reset_index(drop=True)
)

print(f"분석 업종: {target_industry}")
display(industry_detail.round(2))

# 인허가 통합 점포 데이터 EDA 결과

## 1. 분석 개요

점포 단위로 통합한 인허가 데이터를 대상으로 데이터 품질, 업종·지역 분포, 날짜 분포, 기록상 영업 상태를 분석했다.

- 분석 대상: 175,464개 통합점포
- 전체 컬럼: 139개
- 일반 컬럼: 48개
- 업종별 0/1 컬럼: 91개
- 점포 식별 기준: `통합점포ID`
- 영업·폐업 기록이 있는 점포를 모두 포함한 데이터

**기록상 폐업 비중**은 다음과 같이 계산했다.

> 폐업기록 점포 수 ÷ (영업 점포 수 + 폐업기록 점포 수) × 100

이 값은 수집된 데이터의 상태 구성비이며, 연간 폐업률이나 향후 6개월 폐업 확률을 의미하지 않는다.

---

## 2. 데이터 품질 확인

### 2.1 식별자와 중복

- `통합점포ID`의 결측·공란은 0건이다.
- `통합점포ID`의 고유값 수는 175,464개로 전체 행 수와 일치한다.
- 모든 컬럼의 값이 동일한 중복 행도 0건이다.
- 현재 데이터는 `통합점포ID` 기준으로 한 점포당 한 행으로 구성되어 있다.

### 2.2 결측치

결측이 있는 컬럼은 10개이며, 결측의 의미에 따라 처리 방향을 구분해야 한다.

| 컬럼 | 결측 현황 | 해석 및 처리 방향 |
|---|---|---|
| 6개월내폐업_정답 | 100% | 아직 생성하지 않은 정답으로, 예측 기준일 확정 후 생성 |
| 보류사유·추가품질검토사유 | 각각 100% | 관리용 컬럼으로 보관하고 모델 입력에서는 제외 |
| 프랜차이즈_브랜드 | 157,144개, 89.56% | 비프랜차이즈의 ‘해당 없음’과 실제 누락을 구분 |
| 폐업일자·폐업일자_dt·폐업일_후보 | 각각 123,947개, 70.64% | 영업 상태와 대조하여 해석하며 날짜를 임의로 대체하지 않음 |
| 대표인허가ID | 7,043개, 4.01% | 다중 인허가 통합 시 대표 허가를 임의로 선택하지 않은 결과 |
| 법정동 | 196개, 0.11% | 외부 데이터 결합 전 확인 대상 |
| 프랜차이즈_구분 | 1개 | 개별 확인 대상 |

결측률만을 기준으로 컬럼이나 행을 일괄 삭제하지 않았다.

---

## 3. 업종별·구군별 점포 구성

### 3.1 업종별 분포

| 업종 | 점포 수 | 전체 점포 대비 비중 |
|---|---:|---:|
| 일반음식점 | 58,659 | 33.43% |
| 휴게음식점 | 18,274 | 10.41% |
| 미용업 | 17,767 | 10.13% |
| 건강기능식품일반판매업 | 12,435 | 7.09% |
| 즉석판매제조가공업 | 10,672 | 6.08% |

- 일반음식점이 전체의 약 3분의 1을 차지한다.
- 업종별 표본 수가 균등하지 않으므로, 향후 모델 평가에서는 전체 성능과 업종별 성능을 함께 확인할 필요가 있다.
- 복합업종 점포는 7,043개로 전체의 약 4.01%이다.
- 복합업종 점포는 해당 업종마다 집계하므로 업종별 비중의 합은 100%를 넘을 수 있다.
- 상위 업종의 비중을 단순 합산한 값은 해당 업종에 속한 고유 점포의 비중과 다를 수 있다.

### 3.2 구군별 분포

- 부산진구: 22,917개, 13.06%
- 해운대구: 20,912개, 11.92%
- 동래구: 14,030개, 8.00%
- 영도구: 5,118개, 2.92%
- 서구: 4,534개, 2.58%

부산진구와 해운대구를 합하면 43,829개로 전체의 24.98%이다. 가장 많은 부산진구의 점포 수는 가장 적은 서구의 약 5.1배이다.

구군별 점포 수 합계는 전체 행 수인 175,464개와 일치한다.

이 분포는 데이터에 포함된 점포의 구성이며, 현재 영업 중인 점포만의 분포나 면적당 점포 밀도를 의미하지 않는다.

---

## 4. 인허가일·폐업일의 날짜 분포

### 4.1 날짜 품질

- 날짜 변환 실패: 0건
- 최근인허가일이 최초인허가일보다 빠른 사례: 0건
- 폐업일이 최근인허가일보다 빠른 사례: 0건
- 폐업 상태인데 폐업일이 없는 사례: 0건
- 영업 상태인데 폐업일이 있는 사례: 0건

날짜 범위는 다음과 같다.

- 최초인허가일: 1957-01-01 ~ 2026-10-14
- 최근인허가일: 1957-01-01 ~ 2026-10-14
- 폐업일자: 2022-01-01 ~ 2026-10-15

인허가 발생 시점과 데이터 수집 시점은 구분해야 한다. 또한 최초인허가일은 실제 점포 개업일과 다를 수 있다.

### 4.2 연도별 분포

| 연도 | 최초인허가 점포 수 | 폐업기록 점포 수 |
|---|---:|---:|
| 2022 | 13,102 | 9,573 |
| 2023 | 11,408 | 10,676 |
| 2024 | 10,229 | 11,626 |
| 2025 | 9,737 | 11,791 |
| 2026 | 8,387 | 7,851 |

자료상 2023~2025년에는 최초인허가 기록이 감소하고 폐업 기록은 증가했다.

다만 현재 수집된 점포들의 기록 분포이므로, 실제 부산 전체의 개업 감소나 상권 위축을 확정하는 근거로 사용하기에는 수집 대상과 과거 기록의 포함 범위를 더 확인해야 한다.

2026년은 연간 자료가 완성되지 않았으므로 이전 연도의 연간 합계와 직접 비교하지 않는다.

### 4.3 월별 분포

월별 그래프에서는 연말에 폐업 기록이 집중되는 패턴이 관찰되었다.

- 2025년 11월 폐업 기록: 1,026개
- 2025년 12월 폐업 기록: 1,641개
- 2026년 10월 최초인허가 기록: 518개
- 2026년 10월 폐업 기록: 476개

연말 집중이 실제 영업의 계절성인지, 행정상 일괄 처리나 날짜 등록 방식의 영향인지는 추가 확인이 필요하다. 특히 12월 31일 등 특정 날짜에 기록이 몰리는지 점검할 필요가 있다.

10월은 월 전체가 수집되었는지 확인되지 않았으므로 낮은 기록 수를 감소 추세로 해석하지 않는다.

같은 해의 폐업 점포 수를 최초인허가 점포 수로 나눈 값은 폐업률이 아니다. 두 집계에는 서로 다른 시기에 인허가받은 점포들이 포함된다.

---

## 5. 영업 상태와 기록상 폐업 비중

### 5.1 전체 상태

| 상태 | 점포 수 | 비중 |
|---|---:|---:|
| 영업 기록 | 123,947 | 70.64% |
| 폐업 기록 | 51,517 | 29.36% |
| 상태 미확인 | 0 | 0.00% |

전체 기록상 폐업 비중은 29.36%이다.

### 5.2 구군별 차이

- 해운대구: 36.47%
- 기장군: 31.36%
- 연제구: 31.33%
- 부산진구: 29.30%
- 중구: 24.96%
- 영도구: 24.74%

해운대구는 전체보다 7.11%p 높고, 가장 낮은 영도구와는 11.73%p 차이가 난다.

점포 수가 가장 많은 부산진구의 비중은 전체와 비슷하다. 점포 수의 크기와 폐업 기록의 비중은 서로 다른 특성이다.

### 5.3 업종별 차이

점포 수 상위 20개 업종 중 주요 결과는 다음과 같다.

| 업종 | 전체 점포 수 | 기록상 폐업 비중 |
|---|---:|---:|
| 건강기능식품일반판매업 | 12,435 | 57.56% |
| 즉석판매제조가공업 | 10,672 | 49.56% |
| 휴게음식점 | 18,274 | 41.80% |
| 제과점영업 | 1,845 | 39.57% |
| 일반음식점 | 58,659 | 29.37% |
| 미용업 | 17,767 | 23.13% |
| 의원 | 5,758 | 11.11% |
| 쓰레기종량제봉투판매업 | 4,321 | 1.78% |

일반음식점은 폐업기록 건수가 가장 많지만, 비중은 전체와 비슷하다. 반면 건강기능식품일반판매업·즉석판매제조가공업·휴게음식점은 기록상 폐업 비중이 높게 나타났다.

이 차이를 해석할 때는 등록 시기, 관측 기간, 업종별 신고·자료 처리 방식 등을 함께 확인해야 한다.

---

## 6. 구군×업종 교차 분석

### 6.1 비교 기준

- 전체 구군×업종 조합: 1,456개
- 상태 확인 점포가 100개 이상인 조합: 246개
- 히트맵 표시 대상: 점포 수 상위 10개 업종
- 100개 미만 조합은 그림에서 빈칸으로 표시

빈칸은 0%를 의미하지 않는다. 전체 집계 결과는 보존하고, 그래프의 비교 대상만 제한했다.

### 6.2 일반음식점의 구군별 차이

일반음식점 전체의 기록상 폐업 비중은 29.37%이다.

| 구군 | 점포 수 | 기록상 폐업 비중 |
|---|---:|---:|
| 해운대구 | 6,928 | 39.25% |
| 강서구 | 2,976 | 32.86% |
| 기장군 | 3,704 | 32.05% |
| 동구 | 1,948 | 31.98% |
| 부산진구 | 7,513 | 27.49% |
| 북구 | 3,134 | 24.79% |
| 중구 | 2,741 | 22.22% |

일반음식점으로 업종을 좁혀도 해운대구의 높은 비중이 유지된다. 해운대구는 일반음식점 전체보다 9.88%p 높고, 중구와는 17.03%p 차이가 난다.

일반음식점은 모든 구군에서 100개 기준을 충족하며, 구군별 점포 수는 1,526~7,513개이다.

동구는 지역 전체 비중이 26.79%였지만 일반음식점에서는 31.98%로 해당 업종 전체보다 높았다. 따라서 지역 전체 비중만으로 그 지역의 모든 업종을 설명하기 어렵다.

### 6.3 히트맵에서 관찰한 패턴

- 건강기능식품일반판매업과 즉석판매제조가공업은 여러 구군에서 높은 비중을 보인다.
- 의원은 여러 구군에서 상대적으로 낮은 비중을 보인다.
- 해운대구는 일반음식점·휴게음식점·즉석판매제조가공업에서 높은 비중을 보이지만, 모든 업종에서 높게 나타나는 것은 아니다.
- 업종별 공통 패턴과 같은 업종 안의 지역 차이가 함께 관찰된다.
- 쓰레기종량제봉투판매업은 여러 구군에서 0.0%가 반복되고 중구는 23.3%로 나타나, 기록의 수집·처리 차이를 점검할 필요가 있다.

---

## 7. 해석의 한계와 후속 검토

현재 결과는 수집된 데이터의 기술통계이며, 특정 지역·업종의 폐업 원인이나 향후 위험을 확정하는 결과가 아니다.

후속 검토 항목은 다음과 같다.

1. 최초인허가 연도가 비슷한 점포끼리 비교해도 업종·지역 차이가 유지되는지 확인한다.
2. 연말 폐업 기록이 특정 날짜에 집중되는지 확인한다.
3. 특정 업종·구군에서 반복되는 0% 기록의 수집·처리 방식을 확인한다.
4. 외부 데이터 결합 전 행정동코드·좌표·기준 시점과 결합 단위를 확인한다.
5. 데이터 결합 후 행 수 증가, 통합점포ID 중복, 미매칭 여부를 점검한다.
6. 예측 기준일과 완전한 관측 범위를 확정한 뒤 향후 6개월 폐업 정답을 생성한다.
7. 모델 입력에는 예측 기준일 당시 알 수 있었던 정보만 사용한다.

분석 결과 출처: [Colab 분석 노트북](https://colab.research.google.com/drive/1tglxvNEOK5KzAobgxO6EYLQOyl_5uzaT)